# ✍️ `handanim` Interactive Playground (Zero-Install Google Colab)

**Programmatic whiteboard animation in Python** — with auto-synced neural voiceover, procedural chalk & eraser sound effects (`voiceanim`), 2D character inverse kinematics, native LaTeX Beamer/TikZ slide export (`export_beamer()`), and an MCP server for Claude & Cursor (`aianim`).

- 📦 **GitHub Repository**: [https://github.com/subroy13/handanim](https://github.com/subroy13/handanim)
- 📖 **Documentation & Gallery**: [https://subroy13.github.io/handanim/](https://subroy13.github.io/handanim/)

Run each cell below (`Shift + Enter`) to render and play whiteboard animations with stereo audio directly in your browser!

In [ ]:
# 1. Install Cairo C library (on Colab/Ubuntu) + handanim (bundles FFmpeg automatically via imageio-ffmpeg)
!apt-get update -qq && apt-get install -y -qq libcairo2-dev
!pip install -q git+https://github.com/subroy13/handanim.git kokoro-onnx soundfile

import handanim
print("Active Cairo backend:", handanim.get_cairo_backend())

## 🎬 Demo 1: 5-Second Complete Whiteboard Scene (Math + Flowchart + Eraser + Camera Pan + Chalk/Eraser Audio)

With `sketch_sound=True` and `background_music="calm"`, `handanim` automatically synthesizes chalk-on-board sketching sounds, eraser wiping foley, and ducked background music mastered to **-14 LUFS stereo**.

In [ ]:
from IPython.display import Video
from handanim import (
    BLUE, ERASER_HINT_COLOR, NAVY, PASTEL_BLUE, PASTEL_GREEN, PASTEL_ORANGE, RED,
    CameraAnimation, Eraser, FillStyle, Flowchart, Math, Scene, SketchAnimation, StrokeStyle, Text,
)

scene = Scene(width=960, height=540, fps=24, sketch_sound=True, background_music="calm")

formula = Math(r"$y = \sigma(Wx + b)$", position=(80, 70), font_size=64, stroke_style=StrokeStyle(color=BLUE, width=2.5))
draft = Text("Draft: linear only?", position=(80, 215), font_size=36, stroke_style=StrokeStyle(color=RED, width=2))
flow = Flowchart.from_dict({
    "nodes": [
        {"id": "x", "label": "Input x", "position": [220, 390], "size": [190, 74], "font_size": 28, "fill_style": FillStyle(color=PASTEL_BLUE)},
        {"id": "h", "label": "Hidden Wx+b", "position": [540, 390], "size": [230, 74], "font_size": 28, "fill_style": FillStyle(color=PASTEL_ORANGE)},
        {"id": "y", "label": "Output y", "position": [860, 390], "size": [190, 74], "font_size": 28, "fill_style": FillStyle(color=PASTEL_GREEN)},
    ],
    "edges": [
        {"from": "x", "to": "h", "from_side": "right", "to_side": "left"},
        {"from": "h", "to": "y", "from_side": "right", "to_side": "left"},
    ],
})
eraser = Eraser(objects_to_erase=[draft], drawable_cache=scene.drawable_cache, glow_dot_hint={"color": ERASER_HINT_COLOR, "radius": 8})
fixed = Text("Non-linear activation!", position=(80, 215), font_size=38, stroke_style=StrokeStyle(color=NAVY, width=2.2))

scene.add(SketchAnimation(start_time=0.0, duration=1.1), drawable=formula)
scene.add(SketchAnimation(start_time=0.6, duration=0.8), drawable=draft)
scene.add(SketchAnimation(start_time=1.4, duration=1.5), drawable=flow)
scene.add(SketchAnimation(start_time=2.9, duration=0.8), drawable=eraser)
scene.add(SketchAnimation(start_time=3.7, duration=0.6), drawable=fixed)
scene.add(CameraAnimation(start_time=3.6, duration=1.2, data={"to_xrange": (120, 1080), "to_yrange": (40, 580)}))

scene.render("hero_showcase.mp4", max_length=5.0)
Video("hero_showcase.mp4", embed=True, width=720)

## 🎙️ Demo 2: 54-Voice Local Neural TTS, 256-D Latent Voice Blending & Word-Synced Captions (`voiceanim`)

`voiceanim` includes all **54 Kokoro-82M voice embeddings** and lets you blend any combination of voices in 256-D latent style space (e.g. `voice="am_puck:0.6+bm_fable:0.4"` or `voice="blend_transatlantic"`), then trigger drawings on exact spoken words!

In [ ]:
from handanim import Captions, Circle, FillStyle, Math, PASTEL_BLUE, Scene, SketchAnimation, StrokeStyle, BLUE
from voiceanim import VoiceEngine, VoiceTimelineSync

print("Total built-in Kokoro voices:", len(VoiceEngine.list_voices("kokoro")))
print("American male voices:", VoiceEngine.list_voices("kokoro", category="american_male"))

# Synthesize narration using a 60/40 latent blend of am_puck and bm_fable
clip = VoiceEngine.synthesize(
    "Euler's identity connects five fundamental mathematical constants in one equation.",
    voice="am_puck:0.6+bm_fable:0.4",
)

sync = VoiceTimelineSync(clip)
t_start, t_end = sync.word_range("Euler's", "equation.")

scene = Scene(width=960, height=540, fps=24, audio=clip, background_music="marimba", sketch_sound=True)
eq = Math(r"$e^{i\pi} + 1 = 0$", position=(240, 200), font_size=84, stroke_style=StrokeStyle(color=BLUE, width=3))
scene.add(SketchAnimation(start_time=t_start, duration=max(1.2, t_end - t_start)), drawable=eq)

captions = Captions.from_audio_clip(clip, position=(480, 460), font_size=34)
scene.overlay.add(captions.create_animation(), drawable=captions)

scene.render("euler_voice_sync.mp4")
Video("euler_voice_sync.mp4", embed=True, width=720)

## 🧑‍🏫 Demo 3: 2D Character Rigging, 2-Joint Inverse Kinematics & Auto-Blinking (`CharacterRig`)

Build hierarchical stick/whiteboard presenters with `CharacterRig`, scheduled eye blinks (`VariantSlot`), and analytical 2-joint IK (`add_two_joint_limb`).

In [ ]:
from handanim import (
    BLACK, BLUE, CharacterRig, Circle, Curve, FillStyle, Line, MoveBy,
    PASTEL_BLUE, Pose, PoseAnimation, RED, Scene, Show, StrokeStyle,
)

rig = CharacterRig(pivot=(320, 320))
torso = rig.root
torso.drawables.append(Line((320, 220), (320, 320), stroke_style=StrokeStyle(color=BLACK, width=4)))

head = torso.add_child("head", pivot=(320, 220), drawables=[
    Circle(center=(320, 175), radius=36, stroke_style=StrokeStyle(color=BLACK, width=3.5), fill_style=FillStyle(color=PASTEL_BLUE, opacity=0.35))
])
eyes = head.add_slot("eyes", variants={
    "open": [Circle((308, 170), 4), Circle((332, 170), 4)],
    "blink": [Line((303, 170), (313, 170)), Line((327, 170), (337, 170))],
})
eyes.set_schedule([(0.0, "open"), (1.2, "blink"), (1.35, "open"), (2.6, "blink"), (2.75, "open")])

upper_r = torso.add_child("upper_r", pivot=(325, 235), drawables=[Line((325, 235), (385, 235), stroke_style=StrokeStyle(color=BLUE, width=3.5))])
lower_r = upper_r.add_child("lower_r", pivot=(385, 235), drawables=[
    Line((385, 235), (445, 235), stroke_style=StrokeStyle(color=BLUE, width=3.5)),
    Circle((445, 235), 7, stroke_style=StrokeStyle(color=RED, width=3)),
])
rig.add_two_joint_limb("right_arm", "upper_r", "lower_r", effector=(445, 235), bend_direction=1)

scene = Scene(width=720, height=420, fps=24)
scene.add(Show(0.0), drawable=rig)
scene.add(PoseAnimation.from_keyframes(rig, [
    (0.0, Pose(ik_targets={"right_arm": (430, 260)})),
    (1.5, Pose(angles={"head": -8.0}, ik_targets={"right_arm": (440, 145)})),
    (3.0, Pose(angles={"head": 4.0}, ik_targets={"right_arm": (540, 220)})),
]), drawable=rig)
scene.add(MoveBy(dx=100, dy=0, start_time=1.5, duration=1.5), drawable=rig)

scene.render("character_ik_demo.mp4", max_length=3.2)
Video("character_ik_demo.mp4", embed=True, width=640)

## 📄 Demo 4: 1-Line Native LaTeX Beamer & TikZ Slide Deck Export (`export_beamer()`)

Export any animated scene directly into a compilable LaTeX Beamer presentation with inline TikZ vector graphics (`backend="tikz"`).

In [ ]:
from pathlib import Path

tex_path = scene.export_beamer("beamer_slides", n_frames=4, backend="tikz", title="Handanim TikZ Deck")
print("Generated LaTeX Beamer deck at:", tex_path)
print("-" * 60)
print(Path(tex_path).read_text()[:1200] + "\n...")